# Service Access Profiling
Regional Actual Obligations and Service Access. All executable cells use `%sql`.
Only session temporary views are created; no Bronze or Silver tables are written.
**Setup:** Inspect schemas first. Replace the obligations table placeholder and source-column placeholders in the two mapping cells. Optional unavailable fields should use `CAST(NULL AS STRING)`.
The mappings are configurable because the attachment contains no schema. Results must be verified in Databricks.
**BP:** How well does spending align with regional needs and development indicators?
**AP1:** Spending distribution. **AP2:** Budget execution. **AP3:** Spending alignment.
These datasets support AP1 and AP3. Obligations alone cannot measure Allotments → Obligations → Disbursements gaps.


## Inspect Columns

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_obligations;

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.service_access;

## Preview source records

In [0]:
%sql
SELECT * FROM pondolens.dev_bronze.REPLACE_WITH_OBLIGATIONS_TABLE LIMIT 20;

In [0]:
%sql
SELECT * FROM pondolens.dev_bronze.service_access LIMIT 20;

### Configure canonical temporary views
Replace source columns based on DESCRIBE. Keep every grain dimension: extend these views and duplicate keys if the source has additional categories.
For obligations, populate `row_type` (detail/total), `geo_scope` (regional/nationwide/central_office/national_total), `unit_multiplier` and `mapping_verified` only after source validation. `Nationwide` is an allocation bucket, not the national total. Mapping remains blocked by default.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW obligations_profile AS
SELECT
 CAST(`year` AS STRING) AS year_raw,
 TRY_CAST(`year` AS INT) AS reference_year,
 TRIM(`region`) AS region_raw,
 TRIM(`department`) AS entity_name,
 CAST(`value` AS STRING) AS value_raw,
 TRY_CAST(REGEXP_REPLACE(TRIM(CAST(`value` AS STRING)), ',', '') AS DECIMAL(28,6)) AS amount_raw_unit,
 TRIM(`unit`) AS source_unit,
 CAST(`_source_file` AS STRING) AS source_file,
 CAST(NULL AS STRING) AS row_type,
 CAST(NULL AS STRING) AS geo_scope,
 CAST(NULL AS DECIMAL(10,0)) AS unit_multiplier,
 FALSE AS mapping_verified
FROM pondolens.dev_bronze.REPLACE_WITH_OBLIGATIONS_TABLE;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_profile AS
SELECT
 CAST(`year` AS STRING) AS year_raw,
 TRY_CAST(`year` AS INT) AS reference_year,
 TRIM(`region`) AS region_raw,
 TRIM(`indicator`) AS indicator_name,
 CAST(`value` AS STRING) AS value_raw,
 TRY_CAST(REGEXP_REPLACE(TRIM(CAST(`value` AS STRING)), ',', '') AS DECIMAL(28,6)) AS value,
 TRIM(`unit`) AS unit,
 TRIM(`breakdown_type`) AS breakdown_type,
 TRIM(`breakdown_value`) AS breakdown_value,
 CAST(`_source_file` AS STRING) AS source_file,
 CAST(NULL AS STRING) AS missing_reason
FROM pondolens.dev_bronze.service_access;

## 1. Year coverage
A year can have rows but no observed values. Obligations expected 2021–2023; service-access target 2021–2025.

In [0]:
%sql
SELECT reference_year, COUNT(*) AS rows, COUNT(amount_raw_unit) AS observed_values FROM obligations_profile GROUP BY reference_year ORDER BY reference_year;

In [0]:
%sql
WITH expected AS (SELECT EXPLODE(SEQUENCE(2021,2023)) AS reference_year)
SELECT e.reference_year AS missing_year FROM expected e LEFT ANTI JOIN
(SELECT DISTINCT reference_year FROM obligations_profile) p USING (reference_year);

In [0]:
%sql
SELECT reference_year, COUNT(*) AS rows, COUNT(value) AS observed_values FROM service_profile GROUP BY reference_year ORDER BY reference_year;

In [0]:
%sql
WITH expected AS (SELECT EXPLODE(SEQUENCE(2021,2025)) AS reference_year)
SELECT e.reference_year AS missing_year FROM expected e LEFT ANTI JOIN
(SELECT DISTINCT reference_year FROM service_profile) p USING (reference_year);

## 2. Missing and invalid values
Keep NULL observations. Numeric parse failures differ from missing tokens. Confirm negative obligation adjustments with the source.

In [0]:
%sql
SELECT reference_year, source_file, COUNT(*) AS rows,
 SUM(CASE WHEN value_raw IS NULL OR TRIM(value_raw) = '' THEN 1 ELSE 0 END) AS missing_values,
 SUM(CASE WHEN NULLIF(TRIM(value_raw),'') IS NOT NULL AND amount_raw_unit IS NULL THEN 1 ELSE 0 END) AS invalid_numeric_values,
 SUM(CASE WHEN reference_year IS NULL THEN 1 ELSE 0 END) AS invalid_or_missing_years,
 SUM(CASE WHEN NULLIF(TRIM(region_raw),'') IS NULL THEN 1 ELSE 0 END) AS missing_regions
FROM obligations_profile GROUP BY reference_year, source_file ORDER BY reference_year;

In [0]:
%sql
SELECT * FROM obligations_profile WHERE reference_year IS NULL
 OR reference_year NOT BETWEEN 2021 AND 2023
 OR (NULLIF(TRIM(value_raw),'') IS NOT NULL AND amount_raw_unit IS NULL);

In [0]:
%sql
SELECT reference_year, source_file, COUNT(*) AS rows,
 SUM(CASE WHEN value_raw IS NULL OR TRIM(value_raw) = '' THEN 1 ELSE 0 END) AS missing_values,
 SUM(CASE WHEN NULLIF(TRIM(value_raw),'') IS NOT NULL AND value IS NULL THEN 1 ELSE 0 END) AS invalid_numeric_values,
 SUM(CASE WHEN reference_year IS NULL THEN 1 ELSE 0 END) AS invalid_or_missing_years,
 SUM(CASE WHEN NULLIF(TRIM(region_raw),'') IS NULL THEN 1 ELSE 0 END) AS missing_regions
FROM service_profile GROUP BY reference_year, source_file ORDER BY reference_year;

In [0]:
%sql
SELECT * FROM service_profile WHERE reference_year IS NULL
 OR reference_year NOT BETWEEN 2021 AND 2025
 OR (NULLIF(TRIM(value_raw),'') IS NOT NULL AND value IS NULL);

In [0]:
%sql
SELECT * FROM obligations_profile WHERE amount_raw_unit < 0;

In [0]:
%sql
SELECT * FROM service_profile
WHERE LOWER(unit) IN ('%','percent','percentage') AND (value < 0 OR value > 100);

## 3. Duplicate Records
Review conflicting observations and ingestion duplicates before deduplicating. Source file is retained as provenance. Cross-source duplicates require checking definitions and revisions.

In [0]:
%sql
SELECT reference_year, region_raw, entity_name, source_unit, source_file, row_type, COUNT(*) AS duplicate_rows, COUNT(DISTINCT value_raw) AS distinct_nonnull_tokens,
 SUM(CASE WHEN value_raw IS NULL THEN 1 ELSE 0 END) AS null_tokens
FROM obligations_profile GROUP BY reference_year, region_raw, entity_name, source_unit, source_file, row_type HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT reference_year, region_raw, entity_name, source_unit, source_file, row_type, value_raw, COUNT(*) AS repeated_source_observations
FROM obligations_profile GROUP BY reference_year, region_raw, entity_name, source_unit, source_file, row_type, value_raw HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT reference_year, region_raw, indicator_name, unit, breakdown_type, breakdown_value, source_file, COUNT(*) AS duplicate_rows, COUNT(DISTINCT value_raw) AS distinct_nonnull_tokens,
 SUM(CASE WHEN value_raw IS NULL THEN 1 ELSE 0 END) AS null_tokens
FROM service_profile GROUP BY reference_year, region_raw, indicator_name, unit, breakdown_type, breakdown_value, source_file HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT reference_year, region_raw, indicator_name, unit, breakdown_type, breakdown_value, source_file, value_raw, COUNT(*) AS repeated_source_observations
FROM service_profile GROUP BY reference_year, region_raw, indicator_name, unit, breakdown_type, breakdown_value, source_file, value_raw HAVING COUNT(*) > 1;

## 4. Polished Totals
Obligation totals require verified detail rows at a non-overlapping hierarchy level. Preserve source units. Service percentages are never summed across regions or breakdowns.

In [0]:
%sql
SELECT reference_year, source_unit, source_file, geo_scope,
 COUNT(*) AS rows, COUNT(amount_raw_unit) AS observed_amounts,
 SUM(amount_raw_unit) AS total_in_source_unit,
 SUM(amount_raw_unit * unit_multiplier) AS total_php
FROM obligations_profile
WHERE mapping_verified AND row_type = 'detail'
 AND geo_scope IN ('regional','nationwide','central_office')
GROUP BY reference_year, source_unit, source_file, geo_scope ORDER BY reference_year;

In [0]:
%sql
SELECT reference_year, indicator_name, unit, source_file, breakdown_type, breakdown_value,
 COUNT(*) AS rows, COUNT(value) AS observed_values,
 COUNT(*) - COUNT(value) AS missing_or_invalid_values,
 MIN(value) AS minimum_value, MAX(value) AS maximum_value
FROM service_profile
GROUP BY reference_year, indicator_name, unit, source_file, breakdown_type, breakdown_value
ORDER BY reference_year, indicator_name;

## 5. National total vs regional sum
Reconcile per entity/year/unit/source at the same detail level. Include separate Nationwide and Central Office buckets. Do not add hierarchy totals to detail amounts. The national total must be a separate geographic total for that entity.
If totals are in another source, adapt the published-total CTE using its verified year/entity/unit/version keys. Do not equate unmatched sources.
FY2022's previously reported ₱1,000 LGU difference remains unresolved. Exact differences are reported without rounding tolerance.
Service access: not applicable for percentage sums. National estimates require compatible denominators and weights.


In [0]:
%sql
WITH g AS (
 SELECT reference_year, entity_name, source_unit, source_file,
 SUM(CASE WHEN geo_scope = 'regional' THEN amount_raw_unit END) AS regional_sum,
 SUM(CASE WHEN geo_scope = 'nationwide' THEN amount_raw_unit END) AS nationwide_amount,
 SUM(CASE WHEN geo_scope = 'central_office' THEN amount_raw_unit END) AS central_office_amount,
 SUM(CASE WHEN geo_scope IN ('regional','nationwide','central_office') THEN amount_raw_unit END) AS allocated_sum,
 MAX(CASE WHEN geo_scope = 'national_total' THEN amount_raw_unit END) AS published_national_total,
 SUM(CASE WHEN geo_scope = 'national_total' THEN 1 ELSE 0 END) AS national_total_rows,
 SUM(CASE WHEN amount_raw_unit IS NULL THEN 1 ELSE 0 END) AS missing_amount_rows,
 SUM(CASE WHEN NOT mapping_verified OR mapping_verified IS NULL
  OR geo_scope IS NULL OR geo_scope NOT IN ('regional','nationwide','central_office','national_total')
  THEN 1 ELSE 0 END) AS unmapped_rows,
 SUM(CASE WHEN NULLIF(TRIM(region_raw),'') IS NULL THEN 1 ELSE 0 END) AS missing_region_rows,
 COUNT(*) - COUNT(DISTINCT region_raw) AS repeated_geo_rows
 FROM obligations_profile WHERE row_type = 'detail'
 GROUP BY reference_year, entity_name, source_unit, source_file
)
SELECT *, allocated_sum - published_national_total AS difference_raw_unit,
 CASE WHEN reference_year IS NULL OR NULLIF(TRIM(entity_name),'') IS NULL
  OR NULLIF(TRIM(source_unit),'') IS NULL OR NULLIF(TRIM(source_file),'') IS NULL
  OR national_total_rows <> 1 OR missing_amount_rows > 0 OR unmapped_rows > 0
  OR repeated_geo_rows > 0 OR missing_region_rows > 0 OR allocated_sum IS NULL THEN 'INCOMPLETE'
 WHEN allocated_sum = published_national_total THEN 'MATCH' ELSE 'REVIEW' END AS reconciliation_status
FROM g ORDER BY reference_year, entity_name;

## 6. Other relevant checks
Check region boundaries by reference year, source units, missing metadata, and indicator definitions. NCR is not Region XIII; Region XIII is Caraga. Historic 17-region data must not be silently treated as modern 18-region coverage.

In [0]:
%sql
SELECT reference_year, region_raw, COUNT(*) AS rows FROM obligations_profile
GROUP BY reference_year, region_raw ORDER BY reference_year, region_raw;

In [0]:
%sql
SELECT reference_year, indicator_name, source_file, COUNT(DISTINCT region_raw) AS region_labels,
 COUNT(value) AS observed_values FROM service_profile
GROUP BY reference_year, indicator_name, source_file ORDER BY reference_year, indicator_name;

In [0]:
%sql
SELECT DISTINCT indicator_name, unit, source_file, breakdown_type, breakdown_value
FROM service_profile ORDER BY indicator_name;

In [0]:
%sql
SELECT COUNT(*) AS rows, SUM(CASE WHEN NOT mapping_verified OR mapping_verified IS NULL THEN 1 ELSE 0 END) AS unverified_mapping_rows,
 SUM(CASE WHEN row_type IS NULL THEN 1 ELSE 0 END) AS unclassified_hierarchy_rows,
 SUM(CASE WHEN unit_multiplier IS NULL THEN 1 ELSE 0 END) AS unverified_unit_rows
FROM obligations_profile;

## 7. Silver Preparation
PROPOSED. Normalize and cast while retaining raw tokens; validate business keys; use an effective-dated shared region crosswalk; quarantine invalid rows and unresolved mappings. Preserve Nationwide/Central Office scopes.
Keep missing service observations NULL with source-verified reasons. Keep PSA Basic sanitation and JMP At least basic sanitation separate pending comparability checks. Do not fill 2025 observations with 2024 values; any context comparison should carry both years explicitly.
Resolve obligation hierarchy, source units and reconciliation issues before publication. No automatic deduplication. The SQL below previews proposed typed fields, not a production Silver table.


In [0]:
%sql
SELECT reference_year, region_raw, entity_name, geo_scope, source_unit, source_file,
 value_raw, amount_raw_unit,
 CASE WHEN mapping_verified AND unit_multiplier > 0
 THEN amount_raw_unit * unit_multiplier END AS actual_obligations_php,
 CASE WHEN reference_year IS NULL THEN 'INVALID_YEAR'
 WHEN NULLIF(TRIM(value_raw),'') IS NULL THEN 'MISSING_VALUE'
 WHEN amount_raw_unit IS NULL THEN 'INVALID_VALUE'
 WHEN NOT mapping_verified OR mapping_verified IS NULL THEN 'PENDING_MAPPING'
 ELSE 'PENDING_FINAL_VALIDATION' END AS quality_status
FROM obligations_profile LIMIT 100;

In [0]:
%sql
SELECT reference_year, region_raw, indicator_name, unit, breakdown_type, breakdown_value,
 source_file, value_raw, value, missing_reason,
 CASE WHEN reference_year IS NULL THEN 'INVALID_YEAR'
 WHEN NULLIF(TRIM(value_raw),'') IS NULL THEN 'MISSING_VALUE'
 WHEN value IS NULL THEN 'INVALID_VALUE' ELSE 'PENDING_DOMAIN_VALIDATION' END AS quality_status
FROM service_profile LIMIT 100;

## 8. Silver Columns
| Shared column | Type | Purpose |
|---|---|---|
| reference_year | INT | Original observation/accounting year |
| region_code, region_name | STRING | Validated shared crosswalk |
| geo_scope, boundary_version | STRING | Geographic scope and effective boundaries |
| source_id, source_file, source_url, source_page_or_table | STRING | Source/version lineage |
| source_cutoff_date | DATE | Verified cutoff |
| year_raw, region_raw, value_raw | STRING | Original tokens |
| quality_status, issue_code | STRING | Validation flags |
| ingested_at | TIMESTAMP | Preserve Bronze ingestion timestamp |

| Obligations addition | Type | Purpose |
|---|---|---|
| entity_code, entity_name, entity_type | STRING | Department/SPF identity |
| entity_level, row_type | STRING | Validated hierarchy |
| actual_obligations_raw_unit | DECIMAL(28,6) | Source amount |
| source_unit | STRING | Verified monetary unit |
| unit_multiplier | DECIMAL(10,0) | Source unit to pesos |
| actual_obligations_php | DECIMAL(38,6) | Converted obligation amount |

No expense class, sector, or current-year GAA attribution is inferred from this table.

| Service addition | Type | Purpose |
|---|---|---|
| indicator_code, indicator_name | STRING | Source-specific indicator |
| indicator_definition, denominator_description | STRING | Meaning and population base |
| value | DECIMAL(28,6) | Observed value; NULL preserved |
| unit, breakdown_type, breakdown_value | STRING | Unit and breakdown |
| survey_or_method, comparability_group | STRING | Comparison constraints |
| missing_reason | STRING | Verified reason or unverified |

Record findings and blockers after execution. Current outputs have not been executed against Databricks. These are proposed schema columns; crosswalks and metadata still need implementation.
